# BR06A — Leakage-guarded final-CXR k sensitivity

Use this **instead of the generation section of BR06** when testing k=1–5. Run after BR02/BR03 completes. It uses BR06's shared pools and selection-only child runs; no additional CXR extraction is required. Each child gets its own leakage audit and guarded reports. Start on validation, fix k before testing external performance, and keep the common context budget.

Run once per generator in its working environment. Different generator folders can run independently; do not run the same folder from BR04/BR06 at the same time. For evaluation and common-query summaries, return to BR06's evaluation cells using the same `CONFIGS` and child folders.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
from extensions.k_sweep import prepare_sweep
from extensions.leakage_controls import audit_packets
K_VALUES = [1, 2, 3, 4, 5]
SWEEP_VERSION = 'v1'
children = prepare_sweep(CFG, values=K_VALUES, sweep_version=SWEEP_VERSION)
CONFIGS = {k: item['config'] for k, item in children.items()}
for k, config in CONFIGS.items():
    check = audit_packets(config, max_run_hours=1.0)
    print('k=', k, 'passed=', check['passed'], '|', config['run_dir'])
    if not check['passed']:
        raise RuntimeError(f'Leakage gate failed for k={k}; inspect that child audit.')

In [ ]:
from generation import model_snapshot, load_generator
MODEL_KEY = 'pulse'  # pulse, llama_ecg, llama_base, qwen38
MODEL_PATH = None
DOWNLOAD_MODEL = False  # Optional authenticated Hugging Face download into existing cache.
TOKEN_FILE = '/path/to/private/hf_token'  # First line only; never printed.
ENABLE_THINKING = False  # Only Qwen has this switch; primary experiment is False.
LOAD_IN_4BIT = False
PULSE_REPO = None  # Your working PULSE repository when MODEL_KEY='pulse'.
MODEL_PATH = model_snapshot(MODEL_KEY, MODEL_PATH, download=DOWNLOAD_MODEL,
                            token_file=TOKEN_FILE, pin_dir=RUN / 'model_downloads')
runtime = load_generator(CFG, MODEL_KEY, MODEL_PATH, enable_thinking=ENABLE_THINKING,
                         load_in_4bit=LOAD_IN_4BIT, pulse_repo=PULSE_REPO)
print('Loaded:', runtime['model_id'], runtime['revision'])


In [ ]:
from extensions.leakage_controls import guarded_generate_sweep
from generation import folder_for
MAX_NEW_PER_K = 5  # Pilot across k, then 0 for remaining reports.
RETRY_ERRORS = False
MAX_RUN_HOURS = 30.0  # One total budget across all children, including their audits.
MAX_NEW_TOKENS = 1024 if MODEL_KEY == 'qwen38' else 384
print(guarded_generate_sweep(CONFIGS, runtime, max_new_per_k=MAX_NEW_PER_K,
    retry_errors=RETRY_ERRORS, max_run_hours=MAX_RUN_HOURS, max_new_tokens=MAX_NEW_TOKENS))
for k, config in CONFIGS.items():
    print(k, folder_for(config, MODEL_KEY, ENABLE_THINKING, MAX_NEW_TOKENS) / 'reports.jsonl')